In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import feature_extraction as fe 
np.random.seed(42)

In [ ]:
IMAGES_DIR = "ISIC2018_Task3_Training_Input"          # pasta com os .jpg
MASKS_DIR  = "ISIC2018_Task1_Training_GroundTruth"     # opcional (ou None)
LABELS_CSV = "ISIC2018_Task3_Training_GroundTruth.csv" # one-hot das 7 classes
FEATURES_NPZ = "isic_features.npz"

USE_SHAPE = os.path.isdir(MASKS_DIR) if MASKS_DIR else False
print("Usar forma (máscaras encontradas):", USE_SHAPE)

In [ ]:
if not os.path.exists(FEATURES_NPZ):
    cfg = fe.FeatureConfig(use_shape=USE_SHAPE)
    fe.build_feature_database(
        images_dir=IMAGES_DIR,
        masks_dir=MASKS_DIR if USE_SHAPE else None,
        cfg=cfg,
        out_path=FEATURES_NPZ,
    )
else:
    print(f"{FEATURES_NPZ} já existe — pulando extração.")

Carregar


In [ ]:
data = np.load(FEATURES_NPZ, allow_pickle=True)
ids = data["ids"]
blocks = {k: data[k] for k in data.files if k != "ids"}
print("Blocos:", {k: v.shape for k, v in blocks.items()})
print("N imagens:", len(ids))

In [ ]:
# Rótulos: CSV one-hot do Task 3 -> nome da classe por id
gt = pd.read_csv(LABELS_CSV)
class_cols = [c for c in gt.columns if c != "image"]
gt["label"] = gt[class_cols].values.argmax(axis=1)
id2label = dict(zip(gt["image"], gt["label"]))

labels = np.array([id2label.get(i, -1) for i in ids])
mask_valid = labels >= 0
print("Com rótulo:", mask_valid.sum(), "/", len(labels))
print("Distribuição:", dict(zip(*np.unique(labels[mask_valid], return_counts=True))))

Normalização

In [ ]:
from sklearn.preprocessing import StandardScaler

def normalize_block(name, X):
    if name in ("color", "lbp"):      # histogramas: mantém distribuição
        return X
    return StandardScaler().fit_transform(X)

norm_blocks = {k: normalize_block(k, v) for k, v in blocks.items()}

Matrizes

In [ ]:
from sklearn.metrics import pairwise_distances

def chi2_distance(X):
    # distância qui-quadrado vetorizada entre todas as linhas
    n = X.shape[0]
    D = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        num = (X[i] - X) ** 2
        den = (X[i] + X) + 1e-10
        D[i] = 0.5 * (num / den).sum(axis=1)
    return D

METRIC = {"deep": "cosine", "color": "chi2", "lbp": "chi2",
          "gabor": "euclidean", "glcm": "euclidean", "shape": "euclidean"}

def block_distance(name, X):
    m = METRIC.get(name, "euclidean")
    D = chi2_distance(X) if m == "chi2" else pairwise_distances(X, metric=m)
    dmin, dmax = D.min(), D.max()
    return (D - dmin) / (dmax - dmin + 1e-10)   # -> [0,1]

dist = {k: block_distance(k, v) for k, v in norm_blocks.items()}
print("Matrizes de distância prontas:", list(dist.keys()))

Fusão tardia e recuperação

In [ ]:
DEFAULT_WEIGHTS = {"deep": 0.5, "color": 0.2, "gabor": 0.1,
                   "glcm": 0.1, "lbp": 0.05, "shape": 0.05}

def fuse(weights):
    keys = [k for k in weights if k in dist]
    w = np.array([weights[k] for k in keys])
    w = w / w.sum()
    D = sum(wi * dist[k] for wi, k in zip(w, keys))
    return D

def retrieve(query_idx, D, k=10):
    order = np.argsort(D[query_idx])
    order = order[order != query_idx]   # remove a própria consulta
    return order[:k]

Visualizar uma consulta

In [ ]:
CLASS_NAMES = class_cols  # nomes das colunas do CSV

def show_query(query_idx, D, k=8):
    res = retrieve(query_idx, D, k)
    fig, axes = plt.subplots(1, k + 1, figsize=(2.2 * (k + 1), 2.6))
    def load(i):
        return np.array(Image.open(os.path.join(IMAGES_DIR, ids[i] + ".jpg")))
    axes[0].imshow(load(query_idx)); axes[0].set_title(
        f"CONSULTA\n{CLASS_NAMES[labels[query_idx]]}", color="navy")
    axes[0].axis("off")
    for ax, ri in zip(axes[1:], res):
        ax.imshow(load(ri))
        ok = labels[ri] == labels[query_idx]
        ax.set_title(CLASS_NAMES[labels[ri]], color="green" if ok else "red")
        ax.axis("off")
    plt.tight_layout(); plt.show()

D_fused = fuse(DEFAULT_WEIGHTS)
q = np.random.choice(np.where(mask_valid)[0])
show_query(q, D_fused)

Avaliação

In [ ]:
def average_precision(rel):
    # rel: array booleano de relevância na ordem recuperada
    if rel.sum() == 0:
        return 0.0
    hits = np.cumsum(rel)
    prec_at_i = hits / (np.arange(len(rel)) + 1)
    return (prec_at_i * rel).sum() / rel.sum()

def evaluate(D, k=10, query_idxs=None):
    if query_idxs is None:
        query_idxs = np.where(mask_valid)[0]
    p_at_k, r_at_k, aps = [], [], []
    for qi in query_idxs:
        order = retrieve(qi, D, k=D.shape[0] - 1)   # ranking completo p/ mAP
        rel = (labels[order] == labels[qi])
        total_rel = (labels[mask_valid] == labels[qi]).sum() - 1
        p_at_k.append(rel[:k].mean())
        r_at_k.append(rel[:k].sum() / max(total_rel, 1))
        aps.append(average_precision(rel))
    return {"P@%d" % k: np.mean(p_at_k),
            "R@%d" % k: np.mean(r_at_k),
            "mAP": np.mean(aps)}

# amostra de consultas para ir rápido; use todas no resultado final
sample = np.random.choice(np.where(mask_valid)[0],
                          size=min(300, mask_valid.sum()), replace=False)
print(evaluate(D_fused, k=10, query_idxs=sample))